# run69 — priority-XSUBJ: cross-subject pretraining, then per-subject fine-tuning (seed 42)

Subjects are split into 4 fixed groups. For each group, one model is pretrained on the **other ~19 subjects** (2 of them held out for early stopping); each subject of the group is then fine-tuned from it with the unchanged leave-one-session-out protocol. A subject never contributes to the model it starts from (checked twice).

Baseline seed 42: window **0.6074**, trial **0.6325**. Run seeds 999/2024 only if seed 42 improves — and each seed needs **its own pretraining** (`--seed 999`, `--seed 2024`).

## step 1 — pretraining (4 models; resumable: re-running skips finished groups)
Reads the data memory-mapped from disk (~3 GB per subject does not fit in RAM). Prints the folder to use in step 2.

In [ ]:
%run pretrain_cross_subject.py --seed 42

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --cafe-lambda 0

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

## step 2 — fine-tuning from the pretrained models
Replace `<pretrain dir>` with the path printed at the end of step 1 (`...\pretrain_<id>_seed42`). Check the start of the log: `[pretrained] 26 subjects mapped to 4 pretrained models`.

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --pretrained-dir "<pretrain dir>"

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --cafe-lambda 0

## optional step 2b — same, with a lower fine-tuning learning rate
A full-size lr can overwrite what pretraining learned. Uses the same pretrained models (no new pretraining); gets its own results folder.

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --pretrained-dir "<pretrain dir>" --lr 0.0003

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --cafe-lambda 0

## trial-level evaluation
No extra flag: the fine-tuned checkpoints have the run60 architecture.

In [ ]:
%run trial_level_eval.py --seed 42 --variant-hash <hash_step2> --expected-window-acc <grand_step2>

In [ ]:
%run trial_level_eval.py --seed 42 --variant-hash <hash_step2b> --expected-window-acc <grand_step2b>